In [1]:
import pandas as pd

In [2]:
import pickle

In [3]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import root_mean_squared_error

In [4]:
import mlflow


mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("nyc-taxi-experiment")

2025/05/25 23:56:50 INFO mlflow.tracking.fluent: Experiment with name 'nyc-taxi-experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1748217410544, experiment_id='1', last_update_time=1748217410544, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>

In [9]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)
    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
    df = df[(df.duration >=1) & (df.duration <= 60)]
    categorical = ['PULocationID','DOLocationID']
    df[categorical] = df[categorical].astype(str)
    df['PU_DO'] = df['PULocationID'] + df['DOLocationID']
    return df

In [10]:
df_train = read_dataframe('./data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('./data/green_tripdata_2021-02.parquet')

In [13]:
categorical = ['PU_DO']#['PULocationID','DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()
train_dict = df_train[categorical+numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dicts = df_val[categorical+numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [14]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [17]:
import xgboost as xgb 

In [20]:
from pathlib import Path

In [21]:
models_folder = Path('models')
models_folder.mkdir(exist_ok=True)

In [22]:
import mlflow.xgboost
with mlflow.start_run():
    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)

    best_params = {
    'learning_rate': 0.25354536063020344,
    'max_depth':86 ,
    'min_child_weight': 2.9828515568234,
    'objective': 'reg:linear',
    'reg_alpha': 0.028145387058627437,
    'reg_lambda': 0.30613009300449756,
    'seed': 42,
    } 
    mlflow.log_params(best_params)
    booster = xgb.train(
            params=best_params,
            dtrain=train,
            num_boost_round=30,
            evals=[(valid, 'validation')],
            early_stopping_rounds=50
    )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_val,y_pred)
    mlflow.log_metric("rmse", rmse)

    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

/home/codespace/anaconda3/envs/exp-tracking-env/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [00:30:39] WARNING: /workspace/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)


[0]	validation-rmse:10.20700
[1]	validation-rmse:8.86392
[2]	validation-rmse:7.98798
[3]	validation-rmse:7.43305
[4]	validation-rmse:7.08344
[5]	validation-rmse:6.86765
[6]	validation-rmse:6.72986
[7]	validation-rmse:6.63795
[8]	validation-rmse:6.57962
[9]	validation-rmse:6.53592
[10]	validation-rmse:6.50770
[11]	validation-rmse:6.48598
[12]	validation-rmse:6.47165
[13]	validation-rmse:6.45960
[14]	validation-rmse:6.44853
[15]	validation-rmse:6.43969
[16]	validation-rmse:6.43377
[17]	validation-rmse:6.42851
[18]	validation-rmse:6.42277
[19]	validation-rmse:6.41875
[20]	validation-rmse:6.41564
[21]	validation-rmse:6.41419
[22]	validation-rmse:6.41129
[23]	validation-rmse:6.40619
[24]	validation-rmse:6.40379
[25]	validation-rmse:6.40128
[26]	validation-rmse:6.39910
[27]	validation-rmse:6.39717
[28]	validation-rmse:6.39575
[29]	validation-rmse:6.39311


/home/codespace/anaconda3/envs/exp-tracking-env/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [00:30:56] WARNING: /workspace/src/c_api/c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)
2025/05/26 00:30:59 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run thoughtful-seal-858 at: http://localhost:5000/#/experiments/1/runs/67d3ad46eec946f5a1dd7c1899bb2c6e
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [ ]:
logged_model = 'runs:/10731d28ea3d490dbc6a6261fc46308a/models_mlflow'

# Load model as a PyFuncModel.
loaded_model = mlflow.pyfunc.load_model(logged_model)

In [ ]:
loaded_model

In [ ]:
import mlflow.xgboost


xgboost_model = mlflow.xgboost.load_model(logged_model)

In [ ]:
y_pred = xgboost_model.predict(valid)